# Pauli and HSI with PolSARpro (STA1 stack)

Takes one extracted stack of a downloaded zone (`ROI`, `STACK`), builds the averaged coherency matrix T3 of every scene with
the pipeline window (`averaging_window` in `config.yaml`, unchanged), then makes:

* **Pauli RGB** with `polsarpro.util.pauli_rgb`
* **HSI** from `polsarpro.decompositions.h_a_alpha`: hue = mean alpha, saturation = 1 − entropy, intensity = span
* **H-alpha plane** of every scene with `polsarpro.util.plot_h_alpha_plane`

Tune the parameters in the two "parameters" cells and rerun only that cell and the next one.
Kernel: `polsarpro` conda env.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from polsarpro.decompositions import h_a_alpha
from polsarpro.util import pauli_rgb, plot_h_alpha_plane

TOPIC_DIR = next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / "flooded_forest").is_dir())
sys.path.insert(0, str(TOPIC_DIR))

from flooded_forest.core import config
from flooded_forest.core.io import load_stack_info
from flooded_forest.plotting.common import cell_geometry, hsi_to_rgb
from flooded_forest.processing.averaging import AveragingGrid, describe_window
from flooded_forest.processing.polsar import MIN_VALID_FRACTION, averaged_t3, incidence_scale, sinclair

ROI = "Lac"   # zone name (or part of it), one of the downloaded zones listed below
STACK = "STA1"     # stack folder prefix (e.g. "STA1", "STA3_ascending"), one of the stacks listed below

cfg = config.load_config()


def zone_name(folder):
    """ROI name saved by the extraction (the folder name if there is no summary)."""
    summary = folder / "extraction" / "extraction_summary.json"
    return json.loads(summary.read_text(encoding="utf-8"))["roi"]["name"] if summary.exists() else folder.name


zones = {zone_name(d): d for d in sorted(cfg.outputs_dir.iterdir()) if (d / "extraction").is_dir()}
print("downloaded zones:")
for name, folder in zones.items():
    stacks = [s.name for s in cfg.roi_paths(folder.name).stack_dirs()]
    print(f"  {name}  ({folder.name}): {stacks}")

# an exact name (zone or folder, any case) wins; otherwise ROI must be part of exactly one name
matches = [n for n in zones if ROI.lower() in (n.lower(), zones[n].name.lower())]
if not matches:
    matches = [n for n in zones if ROI.lower() in n.lower() or ROI.lower() in zones[n].name.lower()]
if len(matches) != 1:
    raise ValueError(f"ROI {ROI!r} matches {matches or 'no zone'}: choose one of {list(zones)}")
roi_name = matches[0]
stack_dirs = [d for d in cfg.roi_paths(zones[roi_name].name).stack_dirs() if d.name.startswith(STACK)]
if not stack_dirs:
    raise ValueError(f"no stack starting with {STACK!r} in {roi_name}")
stack_dir = stack_dirs[0]
info = load_stack_info(stack_dir)
print(f"\n-> {roi_name} / {stack_dir.name}: {len(info['products'])} scenes")

## Load the scenes: S → averaged T3 (same window and grid as the pipeline)

H / alpha are computed in the next cell, after the water mask.

In [ ]:
win = describe_window(cfg.averaging_window, info["spacings"])
scale = incidence_scale(cfg.incidence_deg(info["swath"]))
with rasterio.open(stack_dir / "overlap_mask.tif") as src:
    inside = src.read(1).astype(bool)
grid = AveragingGrid(inside.shape, win)
geo = cell_geometry(win, grid.az_index, grid.rg_index)

scenes = []
for prod in info["products"]:
    s, finite = sinclair(stack_dir / prod["file"], scale)
    invalid = grid.average((inside & finite).astype(np.float64)) < MIN_VALID_FRACTION
    t3 = averaged_t3(s, finite, grid, invalid)
    span = (t3.m11 + t3.m22 + t3.m33).values
    hh = grid.average(np.abs(s.hh.values) ** 2) / np.maximum(grid.average(finite.astype(np.float64)), 1e-12)
    hh[invalid] = np.nan                                       # |HH|^2 on the same window and grid as T3
    scenes.append({"date": prod["datetime"][:10], "t3": t3,
                   "span_db": 10 * np.log10(np.clip(span, 1e-12, None)),
                   "hh_db": 10 * np.log10(np.clip(hh, 1e-12, None))})
    print(" ", scenes[-1]["date"])
print(f"window {cfg.averaging_window:g} m = {win['NW_a']} x {win['NW_r']} px, grid {grid.shape_out}")


def plot_stack(images, title, cbar=None):
    """One panel per scene, in radar geometry with the true ground aspect; cbar = (cmap, vmin, vmax, label)."""
    fig, axes = plt.subplots(1, len(images), figsize=(2.3 * len(images), 5.2), layout="constrained")
    for ax, img, sc in zip(np.atleast_1d(axes), images, scenes):
        ax.imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
        ax.set_title(sc["date"], fontsize=9)
        ax.tick_params(labelsize=6)
    if cbar:
        cmap, vmin, vmax, label = cbar
        fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(vmin, vmax), cmap), ax=axes, shrink=0.6, label=label)
    fig.suptitle(f"{roi_name} · {stack_dir.name}: {title}", fontsize=10)
    plt.show()

## Water mask (still water removed everywhere)

Still water (river, lakes) is very dark and its polarimetric parameters are mostly noise. A cell
is **water** when its median power over all passes is below `WATER_DB`:

* `WATER_CHANNEL = "span"` (default): total power T11 + T22 + T33 in dB
* `WATER_CHANNEL = "hh"`: HH intensity in dB (removes more: also dark fields)

The water cells are set to NaN in T3, span and HH of **every** pass, before anything else is
computed: they are excluded from H / alpha, Pauli, HSI, H-alpha, the box selection and the change
detection. `WATER_DB = None` switches the mask off. Then H / A / alpha are computed with
`h_a_alpha(t3, boxcar_size=[1, 1])` (T3 is already averaged, so no extra boxcar).

In [ ]:
# --- parameters ---
WATER_DB = -11.0          # cells with median power below this [dB] are water; None = no mask
WATER_CHANNEL = "span"    # "span" (total power) or "hh"
MASK_SCENE = None         # scene shown in the check map; None = a random one

import xarray as xr

key = {"span": "span_db", "hh": "hh_db"}[WATER_CHANNEL]
median_db = np.nanmedian(np.stack([sc[key] for sc in scenes]), axis=0)
water = (median_db < WATER_DB) if WATER_DB is not None else np.zeros(median_db.shape, bool)
cell_km2 = win["output_spacing_az_m"] * win["output_spacing_rg_m"] / 1e6
print(f"water: median {WATER_CHANNEL} < {WATER_DB} dB -> {int(water.sum())} cells = {water.sum() * cell_km2:.1f} km² "
      f"({water.sum() / np.isfinite(median_db).sum():.1%} of the valid cells)")

# check map on one scene (before removing the water): power, water mask, power with the water removed
k_show = int(np.random.default_rng().integers(len(scenes))) if MASK_SCENE is None else MASK_SCENE
power = scenes[k_show][key]
lo, hi = np.nanpercentile(power, (2, 98))
fig, axes = plt.subplots(1, 3, figsize=(10, 5.5), sharex=True, sharey=True, layout="constrained")
im = axes[0].imshow(power, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest", cmap="jet",
                    vmin=lo, vmax=hi)
axes[1].imshow(np.where(np.isfinite(power), water, np.nan), extent=geo["extent"], aspect=geo["px_aspect"],
               interpolation="nearest", cmap="Blues", vmin=0, vmax=1.3)
axes[2].imshow(np.where(water, np.nan, power), extent=geo["extent"], aspect=geo["px_aspect"],
               interpolation="nearest", cmap="jet", vmin=lo, vmax=hi)
for ax, name in zip(axes, (f"{WATER_CHANNEL} [dB]", f"water mask (median < {WATER_DB} dB)", "water removed")):
    ax.set_title(name, fontsize=9)
    ax.tick_params(labelsize=6)
fig.colorbar(im, ax=axes, shrink=0.6, label=f"{WATER_CHANNEL} [dB]")
fig.suptitle(f"{roi_name} · {stack_dir.name} · {scenes[k_show]['date']}: water mask check", fontsize=10)
plt.show()

# remove the water everywhere, then H / A / alpha
keep = xr.DataArray(~water, dims=("y", "x"))
for sc in scenes:
    sc["t3"] = sc["t3"].where(keep)
    sc["span_db"] = np.where(water, np.nan, sc["span_db"])
    sc["hh_db"] = np.where(water, np.nan, sc["hh_db"])
    sc["eig"] = h_a_alpha(sc["t3"], boxcar_size=[1, 1], flags=("entropy", "alpha", "anisotropy", "lambdas")).compute()
print("H / A / alpha computed without the water cells")

## DEM of the stack

The `height` layer of the stack geometry LUT (`geometry_lut.nc`, written by the extraction from the
reference product of the stack), in radar geometry. It covers the whole stack crop, with one LUT
node every `azimuth_step` x `range_step` crop pixels. The black line is the edge of the common area.
Heights are the LUT values in metres; negative values (e.g. in Belém) suggest a height above the
ellipsoid rather than above sea level.

In [ ]:
# --- parameters ---
DEM_PERCENTILES = (1, 99)    # colour scale of the height [m]

import netCDF4

with netCDF4.Dataset(stack_dir / "geometry_lut.nc") as nc:
    height = nc["height"][:].filled(np.nan)
    step_a, step_r = int(nc.azimuth_step), int(nc.range_step)
    off_a, off_r = int(nc.lut_row_offset), int(nc.lut_col_offset)

# LUT node (a, r) = crop pixel (off_a + a * step_a, off_r + r * step_r): same axes as the other maps
dem_extent = [off_r - step_r / 2, off_r + step_r * (height.shape[1] - 0.5),
              off_a + step_a * (height.shape[0] - 0.5), off_a - step_a / 2]
h_lo, h_hi = np.nanpercentile(height, DEM_PERCENTILES)
print(f"height {np.nanmin(height):.1f} to {np.nanmax(height):.1f} m, LUT {height.shape[0]} x {height.shape[1]} nodes "
      f"(every {step_a} az x {step_r} rg pixels)")

fig, ax = plt.subplots(figsize=(5, 7), layout="constrained")
im = ax.imshow(height, extent=dem_extent, aspect=geo["px_aspect"], cmap="jet", vmin=h_lo, vmax=h_hi,
               interpolation="nearest")
ax.contour(inside, levels=[0.5], colors="black", linewidths=1,
           extent=[-0.5, inside.shape[1] - 0.5, inside.shape[0] - 0.5, -0.5], origin="upper")
ax.set_xlabel("range pixel", fontsize=8)
ax.set_ylabel("azimuth pixel", fontsize=8)
ax.tick_params(labelsize=6)
fig.colorbar(im, ax=ax, shrink=0.7, label="height [m]")
ax.set_title(f"{roi_name}\n{stack_dir.name}\nDEM (geometry LUT height)", fontsize=9)
plt.show()

## Mean alpha and HH intensity of every pass

Reference maps, one panel per date, jet colour map:

* **mean alpha** from `h_a_alpha` (0° surface, 45° volume, 90° double bounce), fixed scale 0–90°
* **HH intensity** ⟨|HH|²⟩ in dB, averaged on the same window and grid as T3, colour scale from
  the `HH_PERCENTILES` of all dates (same scale for every date)

In [ ]:
# --- parameters ---
ALPHA_LIMITS = (0, 90)       # colour scale of the mean alpha [deg]
HH_PERCENTILES = (2, 98)     # colour scale of HH [dB], percentiles over all dates


def scalar_images(maps, vmin, vmax, cmap="jet"):
    """Scalar maps -> RGB with a colour map (NaN = white)."""
    out = []
    for m in maps:
        rgb = plt.get_cmap(cmap)(np.clip((m - vmin) / (vmax - vmin), 0, 1))[..., :3]
        rgb[~np.isfinite(m)] = 1.0
        out.append(rgb)
    return out


plot_stack(scalar_images([sc["eig"]["alpha"].values for sc in scenes], *ALPHA_LIMITS),
           "mean alpha", cbar=("jet", *ALPHA_LIMITS, "mean alpha [deg]"))

hh_lo, hh_hi = np.nanpercentile(np.concatenate([sc["hh_db"].ravel() for sc in scenes]), HH_PERCENTILES)
plot_stack(scalar_images([sc["hh_db"] for sc in scenes], hh_lo, hh_hi),
           "HH intensity", cbar=("jet", hh_lo, hh_hi, "⟨|HH|²⟩ [dB]"))

## Pauli RGB

`pauli_rgb(t3, q)`: R = √T22 (double bounce), G = √T33 (volume), B = √T11 (surface), each
channel divided by its own `q` quantile and clipped to 1. `GAMMA` > 1 brightens.

In [ ]:
# --- parameters ---
Q = 1
GAMMA = 1.0

pauli = [np.clip(pauli_rgb(sc["t3"], q=Q).transpose("y", "x", "band").values, 0, 1) ** (1 / GAMMA)
         for sc in scenes]
plot_stack(pauli, f"Pauli RGB (q = {Q}, gamma = {GAMMA})")

## HSI

From the `h_a_alpha` results:

* hue [deg] = `HUE_SCALE` · alpha + `HUE_OFFSET` (pipeline: 4 · alpha + 0)
* saturation = (1 − entropy) ^ `SAT_GAMMA`
* intensity = span in dB stretched between its `SPAN_PERCENTILES` (over all scenes), times `I_MAX`

In [ ]:
# --- parameters ---
HUE_SCALE = 4.0
HUE_OFFSET = 0.0
SAT_GAMMA = 1.0
SPAN_PERCENTILES = (2, 98)
I_MAX = 1.0

lo, hi = np.nanpercentile(np.concatenate([sc["span_db"].ravel() for sc in scenes]), SPAN_PERCENTILES)
hsi = []
for sc in scenes:
    hue = HUE_SCALE * sc["eig"]["alpha"].values + HUE_OFFSET
    sat = (1 - sc["eig"]["entropy"].values) ** SAT_GAMMA
    intensity = np.clip((sc["span_db"] - lo) / (hi - lo), 0, 1) * I_MAX
    hsi.append(np.nan_to_num(hsi_to_rgb(hue, sat, intensity), nan=1.0))
plot_stack(hsi, f"HSI (hue = {HUE_SCALE:g}·alpha + {HUE_OFFSET:g}, sat = (1−H)^{SAT_GAMMA:g}, "
                f"span {lo:.1f} to {hi:.1f} dB, I_max {I_MAX:g})")

## H-alpha plane

`plot_h_alpha_plane(eig, bins, min_pts)`: 2-D histogram of entropy vs mean alpha of every cell,
with the feasible-region boundary curves. Bins with fewer than `MIN_PTS` cells are not drawn.

In [ ]:
# --- parameters ---
BINS = 200
MIN_PTS = 3

for sc in scenes:
    ax, fig = plot_h_alpha_plane(sc["eig"], bins=BINS, min_pts=MIN_PTS)
    fig.set_size_inches(6, 4.2)
    ax.set_title(f"{roi_name} · {stack_dir.name} · {sc['date']}: H-alpha plane", fontsize=9)
    plt.show()

## Select a box in the H-alpha plane and see where those pixels are

Set the box in `BOX = (H_min, H_max, alpha_min, alpha_max)`. The cell draws the box on the
H-alpha plane of every scene (one row, one panel per date), then marks the cells inside the box
in every scene of the stack. The H-alpha panels use the same histogram (`BINS`, `MIN_PTS`, jet
colours) and the same boundary curves as polsarpro `plot_h_alpha_plane`; the curves are taken from it.

Same filter as the H-alpha plot: a cell is kept only if its (H, alpha) histogram bin (`BINS` x
`BINS` bins over H 0–1 and alpha 0–90°, computed per scene) holds at least `MIN_PTS` cells.
Cells in the sparse bins that the plot does not draw are therefore not selected either. Set
`USE_MIN_PTS = False` to select every cell in the box.

The selected cells are shown (red over the span in grey), so you see where they are and how
the selection changes with the date.

In [ ]:
# --- parameters ---

# this works fine for Belem 
BOX = (0.55, 0.78, 45, 62.0)          # H_min, H_max, alpha_min [deg], alpha_max [deg]

# this is for rio negro 1 and 2 
# BOX = (0.4, 0.6, 45, 62.0)          # H_min, H_max, alpha_min [deg], alpha_max [deg]
# BOX = (0.6, 0.9, 30, 45.0)          # H_min, H_max, alpha_min [deg], alpha_max [deg]



COLOR = (1.0, 0.0, 0.0)              # colour of the selected cells
USE_MIN_PTS = True                   # keep only cells whose H-alpha bin has >= MIN_PTS cells (as in the plot)


def dense_bins(h, a, bins, min_pts):
    """True where the cell's (H, alpha) bin has >= min_pts cells (binning of plot_h_alpha_plane)."""
    ok = np.isfinite(h) & np.isfinite(a)
    hc, ac = np.clip(np.nan_to_num(h), 0, 1), np.clip(np.nan_to_num(a), 0, 90)
    counts = np.histogram2d(hc[ok], ac[ok], bins=bins, range=((0, 1), (0, 90)))[0]
    ih = np.minimum((hc * bins).astype(int), bins - 1)
    ia = np.minimum((ac / 90 * bins).astype(int), bins - 1)
    return ok & (counts[ih, ia] >= min_pts)


h_min, h_max, a_min, a_max = BOX
if h_min >= h_max or a_min >= a_max:
    raise ValueError(f"BOX needs H_min < H_max and alpha_min < alpha_max, got {BOX}")
for sc in scenes:
    h, a = sc["eig"]["entropy"].values, sc["eig"]["alpha"].values
    in_box = (h >= h_min) & (h <= h_max) & (a >= a_min) & (a <= a_max)
    sc["selected"] = in_box & dense_bins(h, a, BINS, MIN_PTS) if USE_MIN_PTS else in_box
    sc["in_box"] = int(in_box.sum())

# box on the H-alpha plane of every scene, side by side
ax_ref, fig_ref = plot_h_alpha_plane(scenes[0]["eig"], bins=BINS, min_pts=MIN_PTS)
curves = [(line.get_xdata(), line.get_ydata()) for line in ax_ref.lines]   # polsarpro boundary curves
plt.close(fig_ref)


def h_alpha_points(sc):
    h, a = sc["eig"]["entropy"].values.ravel(), sc["eig"]["alpha"].values.ravel()
    ok = np.isfinite(h) & np.isfinite(a)
    return h[ok].clip(0, 1), a[ok].clip(0, 90)


vmax = max(np.histogram2d(*h_alpha_points(sc), bins=BINS, range=((0, 1), (0, 90)))[0].max() for sc in scenes)
fig, axes = plt.subplots(1, len(scenes), figsize=(2.6 * len(scenes), 3.0), sharey=True, layout="constrained")
for ax, sc in zip(np.atleast_1d(axes), scenes):
    mesh = ax.hist2d(*h_alpha_points(sc), bins=BINS, range=((0, 1), (0, 90)), cmap="jet", cmin=MIN_PTS,
                     vmin=MIN_PTS, vmax=vmax)[3]
    for x, y in curves:
        ax.plot(x, y, color="black", lw=1)
    ax.add_patch(plt.Rectangle((h_min, a_min), h_max - h_min, a_max - a_min, fill=False, edgecolor="black", lw=2))
    ax.set_title(f"{sc['date']}\n{int(sc['selected'].sum())} cells selected", fontsize=8)
    ax.set_xlabel("Entropy (H)", fontsize=8)
    ax.tick_params(labelsize=6)
np.atleast_1d(axes)[0].set_ylabel("Alpha (°)", fontsize=8)
fig.colorbar(mesh, ax=axes, shrink=0.8, label="Number of points")
fig.suptitle(f"{roi_name} · {stack_dir.name}: H-alpha planes, box H {h_min:g}–{h_max:g}, "
             f"alpha {a_min:g}–{a_max:g}°", fontsize=10)
plt.show()

# selected cells over every scene
lo_db, hi_db = np.nanpercentile(np.concatenate([s["span_db"].ravel() for s in scenes]), (2, 98))
overlays = []
for s in scenes:
    grey = np.clip((s["span_db"] - lo_db) / (hi_db - lo_db), 0, 1)
    img = np.repeat(np.nan_to_num(grey, nan=1.0)[..., None], 3, axis=-1)
    img[s["selected"]] = COLOR
    overlays.append(img)
    valid = np.isfinite(s["eig"]["entropy"].values).sum()
    print(f"  {s['date']}: {int(s['selected'].sum())} cells selected ({s['selected'].sum() / valid:.1%} of the valid cells), "
          f"{s['in_box'] - int(s['selected'].sum())} of the {s['in_box']} cells in the box dropped by the MIN_PTS filter")
plot_stack(overlays, f"cells with H {h_min:g}–{h_max:g} and alpha {a_min:g}–{a_max:g}° (red) over the span (grey)")

In [ ]:
# number of cells selected by the H-alpha box on each date (run the box cell above first)
dates = [sc["date"] for sc in scenes]
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
ax.plot(dates, [int(sc["selected"].sum()) for sc in scenes], "-o", color="#e0302a", lw=2, ms=6)
ax.set_xlabel("date")
ax.set_ylabel("selected cells (red in the maps)")
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", labelsize=8)
ax.grid(color="#e4e3dd", lw=0.5)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"{roi_name} · {stack_dir.name}\ncells in the H-alpha box H {h_min:g}–{h_max:g}, "
             f"alpha {a_min:g}–{a_max:g}°", fontsize=9)
plt.show()

## River mask from the H-alpha plane

The river (still water) always falls in the bottom-left corner of the H-alpha plane: low entropy,
low alpha (specular surface scattering). `RIVER_BOX = (H_min, H_max, alpha_min, alpha_max)` is
drawn as a **red box** on the H-alpha plane of every pass. A cell is **river** when it falls in the
box in at least `RIVER_MIN_FRACTION` of the passes (0.5 = half of them; a single odd pass does not
remove a cell).

From this cell on, the river cells are set to NaN in T3, H / A / alpha, span and HH of every pass,
so they are **excluded from the change detection** below. The cells above this one keep the full
scene (rerun them after this cell only if you want them without the river too).
`RIVER_BOX = None` switches the mask off.

In [ ]:
# --- parameters ---
RIVER_BOX = (0.0, 1, 0.0, 30.0)   # H_min, H_max, alpha_min [deg], alpha_max [deg]; None = no mask
RIVER_MIN_FRACTION = 0.5             # river = in the box in at least this fraction of the passes
MASK_SCENE = None                    # scene shown in the check map; None = a random one

import xarray as xr

in_river = np.stack([
    np.zeros(sc["eig"]["entropy"].shape, bool) if RIVER_BOX is None else
    (sc["eig"]["entropy"].values >= RIVER_BOX[0]) & (sc["eig"]["entropy"].values <= RIVER_BOX[1])
    & (sc["eig"]["alpha"].values >= RIVER_BOX[2]) & (sc["eig"]["alpha"].values <= RIVER_BOX[3])
    for sc in scenes])                                           # (passes, rows, cols)
river = in_river.mean(axis=0) >= RIVER_MIN_FRACTION if RIVER_BOX is not None else in_river[0]
cell_km2 = win["output_spacing_az_m"] * win["output_spacing_rg_m"] / 1e6
valid_cells = np.isfinite(scenes[0]["eig"]["entropy"].values)
print(f"river: in the box in >= {RIVER_MIN_FRACTION:.0%} of the passes -> {int(river.sum())} cells = "
      f"{river.sum() * cell_km2:.1f} km² ({river.sum() / valid_cells.sum():.1%} of the valid cells)")

# 1) H-alpha plane of every pass with the river box in red
ax_ref, fig_ref = plot_h_alpha_plane(scenes[0]["eig"], bins=BINS, min_pts=MIN_PTS)
curves = [(line.get_xdata(), line.get_ydata()) for line in ax_ref.lines]   # polsarpro boundary curves
plt.close(fig_ref)
pts = []
for sc in scenes:
    h, a = sc["eig"]["entropy"].values.ravel(), sc["eig"]["alpha"].values.ravel()
    ok = np.isfinite(h) & np.isfinite(a)
    pts.append((h[ok].clip(0, 1), a[ok].clip(0, 90)))
vmax = max(np.histogram2d(h, a, bins=BINS, range=((0, 1), (0, 90)))[0].max() for h, a in pts)
fig, axes = plt.subplots(1, len(scenes), figsize=(2.6 * len(scenes), 3.0), sharey=True, layout="constrained")
for ax, sc, (h, a), inr in zip(np.atleast_1d(axes), scenes, pts, in_river):
    mesh = ax.hist2d(h, a, bins=BINS, range=((0, 1), (0, 90)), cmap="jet", cmin=MIN_PTS, vmin=MIN_PTS, vmax=vmax)[3]
    for x, y in curves:
        ax.plot(x, y, color="black", lw=1)
    if RIVER_BOX is not None:
        ax.add_patch(plt.Rectangle((RIVER_BOX[0], RIVER_BOX[2]), RIVER_BOX[1] - RIVER_BOX[0],
                                   RIVER_BOX[3] - RIVER_BOX[2], fill=False, edgecolor="red", lw=2))
    ax.set_title(f"{sc['date']}\n{int(inr.sum())} cells in the box", fontsize=8)
    ax.set_xlabel("Entropy (H)", fontsize=8)
    ax.tick_params(labelsize=6)
np.atleast_1d(axes)[0].set_ylabel("Alpha (°)", fontsize=8)
fig.colorbar(mesh, ax=axes, shrink=0.8, label="Number of points")
fig.suptitle(f"{roi_name} · {stack_dir.name}: river box (red) H {RIVER_BOX[0]:g}–{RIVER_BOX[1]:g}, "
             f"alpha {RIVER_BOX[2]:g}–{RIVER_BOX[3]:g}°" if RIVER_BOX else "no river box", fontsize=10)
plt.show()

# 2) check map: river cells in red over the span of one scene, and the share of passes in the box
k_show = int(np.random.default_rng().integers(len(scenes))) if MASK_SCENE is None else MASK_SCENE
span = scenes[k_show]["span_db"]
lo, hi = np.nanpercentile(span, (2, 98))
img = np.repeat(np.nan_to_num(np.clip((span - lo) / (hi - lo), 0, 1), nan=1.0)[..., None], 3, axis=-1)
img[river] = (1.0, 0.0, 0.0)
share = np.where(valid_cells, in_river.mean(axis=0), np.nan)
fig, axes = plt.subplots(1, 2, figsize=(8.5, 5.5), sharex=True, sharey=True, layout="constrained")
axes[0].imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
axes[0].set_title(f"{scenes[k_show]['date']}: river (red) over the span (grey)", fontsize=9)
im = axes[1].imshow(share, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest", cmap="jet",
                    vmin=0, vmax=1)
axes[1].set_title("share of the passes in the river box", fontsize=9)
fig.colorbar(im, ax=axes[1], shrink=0.6, label="share of passes")
for ax in axes:
    ax.tick_params(labelsize=6)
fig.suptitle(f"{roi_name} · {stack_dir.name}: river mask, {int(river.sum())} cells = {river.sum() * cell_km2:.1f} km²",
             fontsize=10)
plt.show()

# 3) remove the river from every pass (used by the change detection below)
keep = xr.DataArray(~river, dims=("y", "x"))
for sc in scenes:
    sc["t3"] = sc["t3"].where(keep)
    sc["eig"] = sc["eig"].where(keep)
    sc["span_db"] = np.where(river, np.nan, sc["span_db"])
    sc["hh_db"] = np.where(river, np.nan, sc["hh_db"])
print("river cells set to NaN in T3, H / A / alpha, span and HH of every pass")

## PolSAR change detection between subsequent passes

Method of A. Marino, *PolSAR data processing using python* (`PolSAR_Change_SAOCOM_solutions.ipynb`),
applied to every pair of subsequent passes (1→2, 2→3, … up to the last) of this stack. The input
is the averaged coherency matrix T3 of each pass (pipeline window, unchanged), so no extra boxcar
is applied.

For each cell, with **T1** = T3 of the earlier pass and **T2** = T3 of the later pass:

| detector | what is computed | meaning |
|---|---|---|
| **Power difference** | eigen-decomposition of the change matrix Tc = T2 − T1 (Hermitian): λ1 ≥ λ2 ≥ λ3 | λ1 > 0: largest power **gain**, λ3 < 0: largest power **loss** |
| **Power ratio** | eigen-decomposition of A = T1⁻¹ T2: σ1 ≥ σ2 ≥ σ3, then σ < 1 → 1/σ | largest relative increase / decrease (scale invariant) |
| **Trace distance** | tr(T1⁻¹ T2) = σ1 + σ2 + σ3 | 3 = no change |
| **Wishart** (proportional) | W = \|T1\| \|T2\| / \|(T1 + T2)/2\|² (determinants) | 1 = no change, smaller = more change; shown as −ln W (0 = no change) |

The eigenvector **u** of a change eigenvalue tells *which mechanism* changed, through the angles
α = arccos|u₁| and β = arccos(|u₂| / sin α). The change RGB images use the Pauli colour coding:
**B = cos α** (surface), **R = sin α cos β** (double bounce), **G = sin α sin β** (volume), times
√|eigenvalue|, divided by `FACT` × the mean √|eigenvalue| and clipped to 1.

Differences from the original notebook:

* **fully vectorized**: the 3×3 matrices of all cells are built as an `(rows, cols, 3, 3)` array and
  decomposed in one call (`np.linalg.eigh`, `np.linalg.det`), no pixel loops;
* the ratio uses the equivalent Hermitian form: with T1 = L Lᴴ (Cholesky), the eigenvalues of
  T1⁻¹T2 are those of M = L⁻¹ T2 L⁻ᴴ (Hermitian, so real eigenvalues, numerically stable), and the
  eigenvectors are u = L⁻ᴴ w;
* the Wishart denominator is the determinant of the mean (T1 + T2)/2 (the original divides det(T1 + T2)
  by 2: a constant factor, same map);
* `NORM = "stack"` normalizes the RGB images with the mean over **all pairs**, so pairs can be
  compared (`"pair"` = per pair, as the original).

In [ ]:
# --- parameters ---
EPS = 1e-9            # added to the diagonal of T1 and T2 (as the original), avoids singular matrices
FACT_DIFF = 2.0       # contrast of the power-difference RGB (original: 2)
NORM = "stack"        # "stack" (one normalization for all pairs) or "pair" (each pair on its own, as the original)


def t3_matrix(t3):
    """(rows, cols, 3, 3) Hermitian coherency matrices of an averaged T3 dataset, EPS on the diagonal."""
    m = {k: t3[k].values.astype(np.complex128) for k in ("m11", "m22", "m33", "m12", "m13", "m23")}
    T = np.empty(m["m11"].shape + (3, 3), np.complex128)
    T[..., 0, 0], T[..., 1, 1], T[..., 2, 2] = m["m11"] + EPS, m["m22"] + EPS, m["m33"] + EPS
    T[..., 0, 1], T[..., 0, 2], T[..., 1, 2] = m["m12"], m["m13"], m["m23"]
    T[..., 1, 0], T[..., 2, 0], T[..., 2, 1] = m["m12"].conj(), m["m13"].conj(), m["m23"].conj()
    return T


def alpha_beta(u):
    """alpha, beta [rad] of eigenvectors u (..., 3) (normalized here)."""
    u = u / np.linalg.norm(u, axis=-1, keepdims=True)
    alpha = np.arccos(np.clip(np.abs(u[..., 0]), 0, 1))
    with np.errstate(invalid="ignore", divide="ignore"):
        beta = np.arccos(np.clip(np.abs(u[..., 1]) / np.sin(alpha), 0, 1))
    return alpha, np.nan_to_num(beta)


def mechanism(alpha, beta):
    """0 surface (cos a), 1 double bounce (sin a cos b), 2 volume (sin a sin b): largest Pauli component."""
    comp = np.stack([np.cos(alpha), np.sin(alpha) * np.cos(beta), np.sin(alpha) * np.sin(beta)])
    return np.argmax(np.nan_to_num(comp, nan=-1), axis=0)


def change_pair(T1, T2, ok):
    """All detectors of one pair on the valid cells `ok` (vectorized). Returns maps (rows, cols)."""
    out = {k: np.full(ok.shape, np.nan) for k in ("lam1", "lam2", "lam3", "alpha1", "beta1", "alpha2", "beta2",
                                                  "alpha3", "beta3",
                                                  "sig1", "sig3", "a1", "b1", "a3", "b3", "trace", "wishart")}
    A, B = T1[ok], T2[ok]                                    # (N, 3, 3)

    # power difference: eigh of Tc = T2 - T1 (ascending eigenvalues)
    d, v = np.linalg.eigh(B - A)
    al1, be1 = alpha_beta(v[..., :, 2])                      # eigenvector of the largest eigenvalue
    al2, be2 = alpha_beta(v[..., :, 1])                      # eigenvector of the middle eigenvalue
    al3, be3 = alpha_beta(v[..., :, 0])                      # eigenvector of the smallest eigenvalue
    out["lam1"][ok], out["lam2"][ok], out["lam3"][ok] = d[:, 2], d[:, 1], d[:, 0]
    out["alpha2"][ok], out["beta2"][ok] = al2, be2
    out["alpha1"][ok], out["beta1"][ok], out["alpha3"][ok], out["beta3"][ok] = al1, be1, al3, be3

    # power ratio: eigenvalues of T1^-1 T2 via the Hermitian M = L^-1 T2 L^-H (T1 = L L^H)
    Linv = np.linalg.inv(np.linalg.cholesky(A))
    LinvH = np.conj(np.swapaxes(Linv, -1, -2))
    M = Linv @ B @ LinvH
    M = (M + np.conj(np.swapaxes(M, -1, -2))) / 2            # remove rounding asymmetry
    s, w = np.linalg.eigh(M)                                 # ascending, real, > 0
    u = LinvH @ w                                            # eigenvectors of T1^-1 T2
    a1, b1 = alpha_beta(u[..., :, 2])
    a3, b3 = alpha_beta(u[..., :, 0])
    out["sig1"][ok], out["sig3"][ok] = s[:, 2], s[:, 0]
    out["a1"][ok], out["b1"][ok], out["a3"][ok], out["b3"][ok] = a1, b1, a3, b3

    # trace distance and (proportional) Wishart
    out["trace"][ok] = s.sum(axis=1)
    det1, det2 = np.abs(np.linalg.det(A)), np.abs(np.linalg.det(B))
    detm = np.abs(np.linalg.det((A + B) / 2))
    out["wishart"][ok] = det1 * det2 / detm ** 2

    # as the original: ratio eigenvalues < 1 inverted (all >= 1), extreme values set to 0
    for k in ("sig1", "sig3"):
        x = out[k]
        x[x < 1] = 1 / (x[x < 1] + 1e-5)
        x[x > 100 * np.nanmean(x)] = 0
    out["wishart_change"] = -np.log(np.clip(out["wishart"], 1e-12, None))   # 0 = no change
    return out


import time
t0 = time.time()
mats = [t3_matrix(sc["t3"]) for sc in scenes]
okm = [np.isfinite(T).all(axis=(-1, -2)) for T in mats]
pairs = []
for k in range(len(scenes) - 1):
    ok = okm[k] & okm[k + 1]
    res = change_pair(mats[k], mats[k + 1], ok)
    res.update(ok=ok, k=k, label=f"T{k + 1}{k + 2}", dates=(scenes[k]["date"], scenes[k + 1]["date"]))
    pairs.append(res)
    print(f"  {res['label']}  {res['dates'][0]} -> {res['dates'][1]}: {int(ok.sum())} cells, "
          f"median Wishart change {np.nanmedian(res['wishart_change']):.3f}, median trace {np.nanmedian(res['trace']):.3f}")
del mats
print(f"{len(pairs)} pairs in {time.time() - t0:.1f} s (grid {grid.shape_out[0]} x {grid.shape_out[1]})")


def plot_pairs(rows, title, cbar=None):
    """rows = [(row label, [image per pair])]; one column per pair, radar geometry."""
    n = len(pairs)
    fig, axes = plt.subplots(len(rows), n, figsize=(2.3 * n, 4.9 * len(rows)), squeeze=False, layout="constrained")
    for r, (name, images) in enumerate(rows):
        for c, (img, p) in enumerate(zip(images, pairs)):
            ax = axes[r, c]
            ax.imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
            ax.set_title(f"{p['label']}: {p['dates'][0][5:]} → {p['dates'][1][5:]}\n{name}", fontsize=8)
            ax.tick_params(labelsize=6)
    if cbar:
        cmap, vmin, vmax, label = cbar
        fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(vmin, vmax), cmap), ax=axes, shrink=0.5, label=label)
    fig.suptitle(f"{roi_name} · {stack_dir.name}: {title}", fontsize=10)
    plt.show()


def change_rgb(mag, alpha, beta, key, fact):
    """Pauli-coded change RGB: B = cos a, R = sin a cos b, G = sin a sin b, times sqrt|magnitude|, normalized."""
    out = []
    norms = [np.nanmean(np.sqrt(np.abs(p[mag]))) for p in pairs]
    stack_norm = np.nanmean(norms)
    for p, pn in zip(pairs, norms):
        amp = np.sqrt(np.abs(p[mag]))
        rgb = np.stack([np.sin(p[alpha]) * np.cos(p[beta]), np.sin(p[alpha]) * np.sin(p[beta]), np.cos(p[alpha])], -1)
        rgb = np.clip(rgb * amp[..., None] / ((stack_norm if NORM == "stack" else pn) * fact), 0, 1)
        rgb[~p["ok"]] = 1.0
        out.append(np.nan_to_num(rgb, nan=1.0))
    return out

### Power difference RGB of every pair

Colour = mechanism that changed (Pauli coding: **red** double bounce, **green** volume, **blue**
surface), brightness = strength of the change.

* **largest (λ1)**: where and which scattering **appeared / increased** (T2 − T1 > 0)
* **middle (λ2)**: the second change mechanism, gain or loss depending on its sign
  (brightness = √|λ2|)
* **smallest (λ3)**: where and which scattering **disappeared / decreased**

In [ ]:
plot_pairs([("difference: largest λ1 (gain)", change_rgb("lam1", "alpha1", "beta1", "lam1", FACT_DIFF)),
            ("difference: middle λ2", change_rgb("lam2", "alpha2", "beta2", "lam2", FACT_DIFF)),
            ("difference: smallest λ3 (loss)", change_rgb("lam3", "alpha3", "beta3", "lam3", FACT_DIFF))],
           f"power difference RGB (R double bounce, G volume, B surface; FACT {FACT_DIFF:g}, norm {NORM})")

### Change heat map of the whole stack

Sum of the change maps of all pairs: the regions that change the most over the stack. `HEAT_MEASURE`
chooses the change map that is summed:

* `"wishart"` (default): −ln W, 0 = no change, uses the whole matrix (power and mechanism)
* `"trace"`: tr(T1⁻¹T2) − 3, 0 = no change
* `"difference"`: max(λ1, −λ3) = strongest power gain or loss (absolute power)
* `"ratio"`: ln σ1 + ln σ3, strongest relative increase and decrease

Only cells valid in every pair are summed.

In [ ]:
# --- parameters ---
HEAT_MEASURE = "wishart"      # "wishart", "trace", "difference", "ratio"
HEAT_PERCENTILES = (2, 99)    # colour scale of the heat map

MEASURES = {
    "wishart": (lambda p: p["wishart_change"], "−ln W"),
    "trace": (lambda p: p["trace"] - 3, "tr(T1⁻¹T2) − 3"),
    "difference": (lambda p: np.fmax(p["lam1"], -p["lam3"]), "max(λ1, −λ3)"),
    "ratio": (lambda p: np.log(np.clip(p["sig1"], 1, None)) + np.log(np.clip(p["sig3"], 1, None)), "ln σ1 + ln σ3"),
}
measure, measure_label = MEASURES[HEAT_MEASURE]
change = np.stack([measure(p) for p in pairs])                 # (pairs, rows, cols)
all_ok = np.logical_and.reduce([p["ok"] for p in pairs])
heat = np.where(all_ok, change.sum(axis=0), np.nan)
h_lo, h_hi = np.nanpercentile(heat, HEAT_PERCENTILES)

fig, ax = plt.subplots(figsize=(6, 8), layout="constrained")
im = ax.imshow(heat, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest", cmap="jet",
               vmin=h_lo, vmax=h_hi)
fig.colorbar(im, ax=ax, shrink=0.6, label=f"sum over {len(pairs)} pairs of {measure_label}")
ax.set_xlabel("range pixel", fontsize=8)
ax.set_ylabel("azimuth pixel", fontsize=8)
ax.tick_params(labelsize=6)
ax.set_title(f"{roi_name}\n{stack_dir.name}\nchange heat map ({HEAT_MEASURE}, {pairs[0]['dates'][0]} → "
             f"{pairs[-1]['dates'][1]})", fontsize=9)
plt.show()

### Double-bounce binary map of the stack (from the change matrix)

Built from the power-difference change matrix Tc = T2 − T1 of every pair of subsequent passes.
In a pair, a cell has a **double-bounce change** when

* **increase**: λ1 > 0 and its eigenvector is double bounce, or
* **decrease**: λ3 < 0 and its eigenvector is double bounce,

where the eigenvector is double bounce when its largest Pauli component is the double-bounce one
(sin α cos β > cos α and > sin α sin β, red in the difference RGB). The binary map is the **union
over all pairs** (all temporal baselines): **1** if the cell has a double-bounce increase or
decrease in at least `DB_MIN_PAIRS` pairs (1 = any pair), **0** otherwise.

Only **strong** changes count: almost every cell has some small λ1 > 0 and λ3 < 0, and in about
3 cells out of 4 one of them is double bounce, so without a threshold the union over all pairs is
the whole scene (99.8 % of Belém STA3). A gain / loss counts when its strength |λ| is at least the
threshold:

* `DB_MIN_LAMBDA = None` (default): the threshold is the value of the top `DB_TOP_PERCENT` % of all
  λ1 and −λ3 of all pairs pooled (one threshold for the whole stack, so pairs are comparable);
* or set `DB_MIN_LAMBDA` to a fixed value (linear power, same units as T3; 0 = every gain / loss).

Only cells valid in every pair are used. Figures: the double-bounce changes of
every pair (red increase, blue decrease, magenta both), the binary map of the stack, and the
double-bounce change area per pair.

In [ ]:
# --- parameters ---
DB_MIN_LAMBDA = None   # fixed threshold on |λ| (linear power), or None = use DB_TOP_PERCENT
DB_TOP_PERCENT = 5.0   # a gain / loss counts if |λ| is in the top % of all λ1 and −λ3 of all pairs
DB_MIN_PAIRS = 1       # 1 = union: double-bounce change in any pair of the stack

cell_km2 = win["output_spacing_az_m"] * win["output_spacing_rg_m"] / 1e6
db_valid = np.logical_and.reduce([p["ok"] for p in pairs])
strength = np.concatenate([np.concatenate([p["lam1"][db_valid], -p["lam3"][db_valid]]) for p in pairs])
db_thr = DB_MIN_LAMBDA if DB_MIN_LAMBDA is not None else np.percentile(strength, 100 - DB_TOP_PERCENT)
db_gain = np.stack([(mechanism(p["alpha1"], p["beta1"]) == 1) & (p["lam1"] > 0) & (p["lam1"] >= db_thr) & db_valid
                    for p in pairs])                                  # (pairs, rows, cols): λ1 increase
db_loss = np.stack([(mechanism(p["alpha3"], p["beta3"]) == 1) & (p["lam3"] < 0) & (-p["lam3"] >= db_thr) & db_valid
                    for p in pairs])                                  # (pairs, rows, cols): λ3 decrease
db_pair = db_gain | db_loss
db_count = db_pair.sum(axis=0)
db_map = db_valid & (db_count >= DB_MIN_PAIRS)                        # binary map of the stack
rule = (f"double-bounce λ1 gain or λ3 loss, |λ| >= {db_thr:.3g}"
        + (f" (top {DB_TOP_PERCENT:g} %)" if DB_MIN_LAMBDA is None else ""))
print(f"{rule} in >= {DB_MIN_PAIRS} pair(s): {int(db_map.sum())} cells = {db_map.sum() * cell_km2:.1f} km² "
      f"({db_map.sum() / db_valid.sum():.1%} of the cells valid in every pair)")
for p, g, l in zip(pairs, db_gain, db_loss):
    print(f"  {p['label']}: increase {int(g.sum())}, decrease {int(l.sum())}, both {int((g & l).sum())} cells")

span_med = np.nanmedian(np.stack([sc["span_db"] for sc in scenes]), axis=0)
lo, hi = np.nanpercentile(span_med, (2, 98))
grey = np.repeat(np.nan_to_num(np.clip((span_med - lo) / (hi - lo), 0, 1), nan=1.0)[..., None], 3, axis=-1)


def db_overlay(gain, loss):
    """Grey median span with red = increase, blue = decrease, magenta = both."""
    img = grey.copy()
    img[gain] = (1.0, 0.0, 0.0)
    img[loss] = (0.0, 0.3, 1.0)
    img[gain & loss] = (1.0, 0.0, 1.0)
    return img


# 1) double-bounce changes of every pair
plot_pairs([("double-bounce change", [db_overlay(g, l) for g, l in zip(db_gain, db_loss)])],
           f"{rule} (red increase, blue decrease, magenta both)")

# 2) binary map of the stack and 3) double-bounce change area per pair
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 5.5), width_ratios=[1, 1.4], layout="constrained")
img = grey.copy()
img[db_map] = (1.0, 0.0, 0.0)
ax0.imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
ax0.set_title(f"double-bounce change in >= {DB_MIN_PAIRS} pair(s) (red)\n{int(db_map.sum())} cells = "
              f"{db_map.sum() * cell_km2:.1f} km²", fontsize=9)
ax0.tick_params(labelsize=6)
labels_p = [p["label"] for p in pairs]
for masks, color, name in ((db_gain, "#e0302a", "increase (λ1)"), (db_loss, "#2a64d6", "decrease (λ3)"),
                           (db_pair, "#9b30d0", "increase or decrease")):
    ax1.plot(labels_p, [m.sum() * cell_km2 for m in masks], "-o", color=color, lw=2, ms=6, label=name)
ax1.axhline(db_map.sum() * cell_km2, color="#7a7a7a", lw=1, label="binary map (union of the pairs)")
ax1.set_ylabel("double-bounce change area [km²]")
ax1.set_ylim(bottom=0)
ax1.legend(frameon=False)
ax1.grid(color="#e4e3dd", lw=0.5)
ax1.spines[["top", "right"]].set_visible(False)
ax1.set_title("double-bounce change area per pair", fontsize=9)
fig.suptitle(f"{roi_name} · {stack_dir.name}: double-bounce binary map ({rule})", fontsize=10)
plt.show()

### Wetland map, and its H-alpha on every pass

`WETLAND_FROM` chooses the wetland surface:

* `"binary"` (default): the double-bounce binary map above (`db_map`, union of the λ1 / λ3 double-bounce changes);
* `"heat"`: the cells whose heat-map value (sum of the changes over the stack, `HEAT_MEASURE` above)
  is at least the threshold:

* `HEAT_THRESHOLD = None` (default): the threshold is the value of the top `HEAT_TOP_PERCENT` % of the cells,
* or set `HEAT_THRESHOLD` to a fixed value, in the units of the heat-map colour bar.

The first figure shows the heat-map histogram with the threshold and the wetland map (red over the
span in grey). Then the H-alpha plane of the wetland cells on every pass, labelled with the temporal
baseline (days since the first pass), and the median entropy and alpha of the wetland over time
against the rest of the scene.

In [ ]:
# --- parameters ---
WETLAND_FROM = "binary"   # "binary" (double-bounce binary map above) or "heat" (threshold on the heat map)
HEAT_THRESHOLD = None     # fixed threshold on the heat map, or None = use HEAT_TOP_PERCENT
HEAT_TOP_PERCENT = 20.0   # wetland = cells in the top % of the heat map
DEM_VERT_EXAG = 30.0      # vertical exaggeration of the DEM shaded relief
DEM_SHADE = 0.6           # strength of the hillshade on the DEM colours (0 = no shading, 1 = full)

cell_km2 = win["output_spacing_az_m"] * win["output_spacing_rg_m"] / 1e6
if WETLAND_FROM == "binary":
    heat_thr = np.nan
    wetland = db_map & np.isfinite(heat)
    wetland_rule = f"{rule} in >= {DB_MIN_PAIRS} pair(s)"
elif WETLAND_FROM == "heat":
    heat_thr = HEAT_THRESHOLD if HEAT_THRESHOLD is not None else np.nanpercentile(heat, 100 - HEAT_TOP_PERCENT)
    wetland = np.isfinite(heat) & (heat >= heat_thr)
    wetland_rule = f"heat ({HEAT_MEASURE}) >= {heat_thr:.3g}"
else:
    raise ValueError(f"WETLAND_FROM must be 'binary' or 'heat', not {WETLAND_FROM!r}")
others = np.isfinite(heat) & ~wetland
print(f"wetland: {wetland_rule} -> {int(wetland.sum())} cells = {wetland.sum() * cell_km2:.1f} km² "
      f"({wetland.sum() / np.isfinite(heat).sum():.1%} of the cells valid in every pair)")

# 1) wetland map and DEM side by side (plus the heat-map histogram with the threshold, heat only)
heat_mode = WETLAND_FROM == "heat"
fig, axes = plt.subplots(1, 3 if heat_mode else 2, figsize=(15 if heat_mode else 10, 7),
                         width_ratios=[1.3, 1, 1] if heat_mode else [1, 1], layout="constrained")
if heat_mode:
    ax_hist = axes[0]
    ax_hist.hist(heat[np.isfinite(heat)], bins=200, range=tuple(np.nanpercentile(heat, (0, 99.5))), color="#7a7a7a")
    ax_hist.axvline(heat_thr, color="#e0302a", lw=1.5, label=f"threshold {heat_thr:.3g}")
    ax_hist.set_xlabel(f"heat map: sum over {len(pairs)} pairs of {measure_label}")
    ax_hist.set_ylabel("cells")
    ax_hist.legend(frameon=False)
    ax_hist.spines[["top", "right"]].set_visible(False)
ax_wet, ax_dem = axes[-2], axes[-1]
span = np.nanmedian(np.stack([sc["span_db"] for sc in scenes]), axis=0)
lo, hi = np.nanpercentile(span, (2, 98))
img = np.repeat(np.nan_to_num(np.clip((span - lo) / (hi - lo), 0, 1), nan=1.0)[..., None], 3, axis=-1)
img[wetland] = (1.0, 0.0, 0.0)
ax_wet.imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
ax_wet.set_title(f"wetland (red) over the median span (grey)\n{int(wetland.sum())} cells = {wetland.sum() * cell_km2:.1f} km²",
                 fontsize=9)
dem_lo, dem_hi = np.nanpercentile(height, DEM_PERCENTILES)       # h_lo / h_hi are reused by the heat map
# shaded relief (as QGIS hillshade + colour): light from the north-west, slopes from the LUT node spacing
from matplotlib.colors import LightSource
ls = LightSource(azdeg=315, altdeg=45)
dem_filled = np.where(np.isfinite(height), height, np.nanmedian(height))
shade = ls.hillshade(dem_filled, vert_exag=DEM_VERT_EXAG, dx=step_r, dy=step_a * geo["px_aspect"])
colour = plt.get_cmap("jet")(np.clip((dem_filled - dem_lo) / (dem_hi - dem_lo), 0, 1))[..., :3]
relief = colour * (1 - DEM_SHADE + DEM_SHADE * shade[..., None])   # colour multiplied by the hillshade
relief[~np.isfinite(height)] = 1.0
ax_dem.imshow(relief, extent=dem_extent, aspect=geo["px_aspect"], interpolation="nearest")
im = plt.cm.ScalarMappable(plt.Normalize(dem_lo, dem_hi), "jet")
ax_dem.contour(inside, levels=[0.5], colors="black", linewidths=1,
               extent=[-0.5, inside.shape[1] - 0.5, inside.shape[0] - 0.5, -0.5], origin="upper")
ax_dem.set_xlim(ax_wet.get_xlim())
ax_dem.set_ylim(ax_wet.get_ylim())
ax_dem.set_title(f"DEM shaded relief (vertical exaggeration {DEM_VERT_EXAG:g})", fontsize=9)
fig.colorbar(im, ax=ax_dem, shrink=0.6, label="height [m]")
for ax in (ax_wet, ax_dem):
    ax.tick_params(labelsize=6)
fig.suptitle(f"{roi_name} · {stack_dir.name}: wetland map and DEM", fontsize=10)
fig.supxlabel(f"wetland = {wetland_rule}", fontsize=9)
plt.show()

# 2) H-alpha plane of the wetland cells on every pass (temporal baseline = days since the first pass)
ax_ref, fig_ref = plot_h_alpha_plane(scenes[0]["eig"], bins=BINS, min_pts=MIN_PTS)
curves = [(line.get_xdata(), line.get_ydata()) for line in ax_ref.lines]   # polsarpro boundary curves
plt.close(fig_ref)
baseline = [int((np.datetime64(sc["date"]) - np.datetime64(scenes[0]["date"])) / np.timedelta64(1, "D")) for sc in scenes]


def h_alpha_of(cells):
    out = []
    for sc in scenes:
        h, a = sc["eig"]["entropy"].values[cells], sc["eig"]["alpha"].values[cells]
        ok = np.isfinite(h) & np.isfinite(a)
        out.append((h[ok].clip(0, 1), a[ok].clip(0, 90)))
    return out


wet_pts, other_pts = h_alpha_of(wetland), h_alpha_of(others)
vmax = max(np.histogram2d(h, a, bins=BINS, range=((0, 1), (0, 90)))[0].max() for h, a in wet_pts)
fig, axes = plt.subplots(1, len(scenes), figsize=(2.6 * len(scenes), 3.1), sharey=True, layout="constrained")
for ax, sc, b, (h, a) in zip(np.atleast_1d(axes), scenes, baseline, wet_pts):
    mesh = ax.hist2d(h, a, bins=BINS, range=((0, 1), (0, 90)), cmap="jet", cmin=MIN_PTS, vmin=MIN_PTS,
                     vmax=max(vmax, MIN_PTS + 1))[3]
    for x, y in curves:
        ax.plot(x, y, color="black", lw=1)
    ax.set_title(f"{sc['date']} (+{b} d)\nmedian H {np.median(h):.2f}, alpha {np.median(a):.1f}°", fontsize=8)
    ax.set_xlabel("Entropy (H)", fontsize=8)
    ax.tick_params(labelsize=6)
np.atleast_1d(axes)[0].set_ylabel("Alpha (°)", fontsize=8)
fig.colorbar(mesh, ax=axes, shrink=0.8, label="Number of points")
fig.suptitle(f"{roi_name} · {stack_dir.name}: H-alpha of the {int(wetland.sum())} wetland cells on every pass",
             fontsize=10)
plt.show()

# 3) median H and alpha of the wetland vs the rest of the scene over the temporal baseline
fig, (ax_h, ax_a) = plt.subplots(1, 2, figsize=(12, 3.5), layout="constrained")
for ax, j, label in ((ax_h, 0, "entropy H"), (ax_a, 1, "mean alpha [deg]")):
    for pts, color, name in ((wet_pts, "#e0302a", "wetland"), (other_pts, "#7a7a7a", "rest of the scene")):
        q25, q50, q75 = np.array([np.percentile(p[j], (25, 50, 75)) for p in pts]).T
        ax.fill_between(baseline, q25, q75, color=color, alpha=0.18, lw=0)
        ax.plot(baseline, q50, "-o", color=color, lw=2, ms=5, label=f"{name} (median, 25–75 %)")
    ax.set_xticks(baseline, [f"+{b}\n{sc['date'][5:]}" for b, sc in zip(baseline, scenes)], fontsize=7)
    ax.set_xlabel("temporal baseline [days since the first pass] / date")
    ax.set_ylabel(label)
    ax.grid(color="#e4e3dd", lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)
ax_a.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("wetland vs rest of the scene over the stack", fontsize=10)
plt.show()

In [ ]:
# export the double-bounce trend of the wetland (one row per pass) for tidal_validation.ipynb
import pandas as pd


def db10(x):
    return float(10 * np.log10(x)) if x > 0 else np.nan


rows = []
for k, (sc, prod) in enumerate(zip(scenes, info["products"])):
    t3v = {m: sc["t3"][m].values for m in ("m11", "m22", "m33")}
    h, a = sc["eig"]["entropy"].values, sc["eig"]["alpha"].values
    row = {"date": sc["date"], "time_utc": prod["datetime"], "baseline_d": baseline[k]}
    for cells, tag in ((wetland, "wet"), (others, "rest")):
        p = {m: np.nanmean(v[cells]) for m, v in t3v.items()}
        row.update({f"{tag}_T11_db": db10(p["m11"]), f"{tag}_T22_db": db10(p["m22"]), f"{tag}_T33_db": db10(p["m33"]),
                    f"{tag}_T22_share": p["m22"] / (p["m11"] + p["m22"] + p["m33"]),
                    f"{tag}_alpha_median": float(np.nanmedian(a[cells])), f"{tag}_H_median": float(np.nanmedian(h[cells]))})
    rows.append(row)
trend = pd.DataFrame(rows)
trend["roi"], trend["stack"] = roi_name, stack_dir.name
trend["wetland_from"], trend["wetland_rule"] = WETLAND_FROM, wetland_rule
trend["heat_measure"], trend["heat_threshold"], trend["wetland_cells"] = HEAT_MEASURE, heat_thr, int(wetland.sum())

trend_path = cfg.roi_paths(zones[roi_name].name).polsar / stack_dir.name / "wetland_trend.csv"
trend_path.parent.mkdir(parents=True, exist_ok=True)
trend.to_csv(trend_path, index=False)
print(f"saved {trend_path}")
trend[["date", "time_utc", "baseline_d", "wet_T22_db", "wet_T22_share", "wet_alpha_median", "wet_H_median", "rest_T22_db"]]

### Statistics of the scattering changes over time

A cell is **changed** in a pair when its change measure (`HEAT_MEASURE`) is in the top
`CHANGE_PERCENT` % of all cells of all pairs: one threshold for the whole stack, so pairs are
comparable. Then:

1. distribution of the change measure per pair (box plot) and changed area per pair;
2. **which mechanism changed**: for the changed cells, the mechanism of the power-difference
   eigenvectors (largest Pauli component of the eigenvector: surface, double bounce or volume),
   for the gains (λ1 > 0) and for the losses (λ3 < 0), as area per pair;
3. **persistence**: in how many pairs each cell is changed (map and histogram): a cell changed in
   one pair only is an event, a cell changed in many pairs is a dynamic area;
4. the mean Pauli powers (T11, T22, T33 in dB) of the cells changed at least once vs. the stable
   cells, per pass: how the scattering of the changing areas evolves.

In [ ]:
# --- parameters ---
CHANGE_PERCENT = 10.0     # top % of the change measure (all pairs pooled) = "changed"

cell_km2 = win["output_spacing_az_m"] * win["output_spacing_rg_m"] / 1e6
labels_p = [p["label"] for p in pairs]
thr = np.nanpercentile(change[:, all_ok], 100 - CHANGE_PERCENT)
changed = (change >= thr) & all_ok                               # (pairs, rows, cols)
print(f"threshold {measure_label} >= {thr:.3g} (top {CHANGE_PERCENT:g} % of all pairs)")

# 1) distribution per pair and changed area
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
ax0.boxplot([c[all_ok] for c in change], showfliers=False, medianprops=dict(color="#e0302a", lw=2))
ax0.set_xticks(range(1, len(pairs) + 1), labels_p)
ax0.axhline(thr, color="#7a7a7a", lw=1, label=f"change threshold {thr:.3g}")
ax0.set_ylabel(measure_label)
ax0.set_title("change measure per pair (box 25–75 %, whiskers 1.5 IQR)", fontsize=9)
ax0.legend(frameon=False)
ax1.bar(labels_p, [c.sum() * cell_km2 for c in changed], color="#2a64d6")
ax1.set_ylabel("changed area [km²]")
ax1.set_title(f"area above the threshold per pair", fontsize=9)
for ax in (ax0, ax1):
    ax.grid(axis="y", color="#e4e3dd", lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{roi_name} · {stack_dir.name}: change per pair ({HEAT_MEASURE})", fontsize=10)
plt.show()

# 2) mechanism of the gains and losses in the changed cells
MECH = {"surface": "#2a64d6", "double bounce": "#e0302a", "volume": "#2fa84f"}


fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True, layout="constrained")
for ax, (kind, lam, al, be, sign) in zip(axes, (("gain (λ1 > 0)", "lam1", "alpha1", "beta1", 1),
                                                 ("loss (λ3 < 0)", "lam3", "alpha3", "beta3", -1))):
    for m, (name, color) in enumerate(MECH.items()):
        area = [((mechanism(p[al], p[be]) == m) & changed[i] & (sign * p[lam] > 0)).sum() * cell_km2
                for i, p in enumerate(pairs)]
        ax.plot(labels_p, area, "-o", color=color, lw=2, ms=6, label=name)
    ax.set_title(f"power {kind} in the changed cells", fontsize=9)
    ax.set_ylabel("area [km²]")
    ax.grid(color="#e4e3dd", lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0), title="mechanism")
fig.suptitle("which scattering mechanism appears (gain) and disappears (loss) in each pair", fontsize=10)
plt.show()

# 3) persistence: number of pairs in which a cell is changed
n_changed = np.where(all_ok, changed.sum(axis=0), np.nan)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 6), width_ratios=[1, 1.2], layout="constrained")
cmap = plt.get_cmap("jet", len(pairs) + 1)
im = ax0.imshow(n_changed, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest", cmap=cmap,
                vmin=-0.5, vmax=len(pairs) + 0.5)
fig.colorbar(im, ax=ax0, shrink=0.6, ticks=range(len(pairs) + 1), label="pairs with change")
ax0.set_title("in how many pairs each cell changed", fontsize=9)
ax0.tick_params(labelsize=6)
counts = [int((n_changed == n).sum()) for n in range(len(pairs) + 1)]
ax1.bar(range(len(pairs) + 1), [c * cell_km2 for c in counts], color=[cmap(n) for n in range(len(pairs) + 1)])
ax1.set_xlabel("number of pairs with change")
ax1.set_ylabel("area [km²]")
ax1.set_yscale("log")
ax1.set_title("area by number of changes (log scale)", fontsize=9)
ax1.grid(axis="y", color="#e4e3dd", lw=0.5)
ax1.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{roi_name} · {stack_dir.name}: persistence of the change", fontsize=10)
plt.show()

# 4) mean Pauli powers of the changing vs stable cells per pass
dynamic, stable = all_ok & (n_changed >= 1), all_ok & (n_changed == 0)
dates = [sc["date"] for sc in scenes]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True, layout="constrained")
for ax, (m, name, color) in zip(axes, (("m22", "T22 double bounce", "#e0302a"), ("m33", "T33 volume", "#2fa84f"),
                                       ("m11", "T11 surface", "#2a64d6"))):
    for cells, style, lab in ((dynamic, "-o", "changed at least once"), (stable, "--s", "never changed")):
        ax.plot(dates, [10 * np.log10(np.nanmean(sc["t3"][m].values[cells])) for sc in scenes], style,
                color=color, lw=2, ms=5, label=lab, alpha=1 if style == "-o" else 0.6)
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis="x", labelsize=7, rotation=30)
    ax.grid(color="#e4e3dd", lw=0.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=7)
axes[0].set_ylabel("mean power [dB]")
fig.suptitle(f"mean Pauli powers per pass: {int(dynamic.sum())} cells changed at least once vs "
             f"{int(stable.sum())} never changed", fontsize=10)
plt.show()

### Animations: H-alpha evolution over the stack (GIF)

Two looping GIFs, one frame per pass, saved next to `wetland_trend.csv` and shown below:

1. **whole scene**: H-alpha plane of every cell (same histogram as the H-alpha plots: `BINS`,
   `MIN_PTS`, jet, polsarpro boundary curves) with the selection `BOX` (black), and on the right the
   cells selected by the box on that pass (red over the span in grey);
2. **wetland only**: the wetland binary map used as mask (left, fixed), the H-alpha plane of the
   wetland cells only (middle) and the Pauli RGB of the pass (right).

The H-alpha colour scale is fixed over all frames, so the density can be compared between passes.
Note: H / alpha are taken after the river mask, so the river cells are not in the histograms.

In [ ]:
# --- parameters ---
GIF_FPS = 1.0     # frames (passes) per second

from IPython.display import Image, display
from matplotlib.animation import FuncAnimation, PillowWriter

gif_dir = cfg.roi_paths(zones[roi_name].name).polsar / stack_dir.name
gif_dir.mkdir(parents=True, exist_ok=True)
ax_ref, fig_ref = plot_h_alpha_plane(scenes[0]["eig"], bins=BINS, min_pts=MIN_PTS)
curves = [(line.get_xdata(), line.get_ydata()) for line in ax_ref.lines]   # polsarpro boundary curves
plt.close(fig_ref)
bh0, bh1, ba0, ba1 = BOX


def draw_h_alpha(ax, h, a, vmax, title):
    """H-alpha histogram as plot_h_alpha_plane (jet, cmin = MIN_PTS) with the boundary curves."""
    ax.clear()
    ax.hist2d(h, a, bins=BINS, range=((0, 1), (0, 90)), cmap="jet", cmin=MIN_PTS, vmin=MIN_PTS,
              vmax=max(vmax, MIN_PTS + 1))
    for x, y in curves:
        ax.plot(x, y, color="black", lw=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 90)
    ax.set_xlabel("Entropy (H)", fontsize=8)
    ax.set_ylabel("Alpha (°)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title(title, fontsize=9)


def save_gif(fig, update, name):
    anim = FuncAnimation(fig, update, frames=len(scenes), interval=1000 / GIF_FPS)
    out = gif_dir / name
    anim.save(out, writer=PillowWriter(fps=GIF_FPS))
    plt.close(fig)
    print(f"saved {out}")
    display(Image(filename=str(out)))


span_all = np.nanmedian(np.stack([sc["span_db"] for sc in scenes]), axis=0)
g_lo, g_hi = np.nanpercentile(np.concatenate([sc["span_db"].ravel() for sc in scenes]), (2, 98))

# 1) whole scene: H-alpha with the BOX, and the cells selected by the box
all_pts = h_alpha_of(np.ones(span_all.shape, bool))
vmax_all = max(np.histogram2d(h, a, bins=BINS, range=((0, 1), (0, 90)))[0].max() for h, a in all_pts)
fig, (ax_ha, ax_map) = plt.subplots(1, 2, figsize=(10, 6), width_ratios=[1.4, 1], layout="constrained")
fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(MIN_PTS, vmax_all), "jet"), ax=ax_ha, shrink=0.7,
             label="Number of points")


def update_scene(k):
    sc, b, (h, a) = scenes[k], baseline[k], all_pts[k]
    draw_h_alpha(ax_ha, h, a, vmax_all, f"{sc['date']} (+{b} d): H-alpha of the whole scene")
    ax_ha.add_patch(plt.Rectangle((bh0, ba0), bh1 - bh0, ba1 - ba0, fill=False, edgecolor="black", lw=2))
    grey = np.clip((sc["span_db"] - g_lo) / (g_hi - g_lo), 0, 1)
    img = np.repeat(np.nan_to_num(grey, nan=1.0)[..., None], 3, axis=-1)
    img[sc["selected"]] = COLOR
    ax_map.clear()
    ax_map.imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
    ax_map.set_title(f"cells in the box (red): {int(sc['selected'].sum())}", fontsize=9)
    ax_map.tick_params(labelsize=6)
    fig.suptitle(f"{roi_name} · {stack_dir.name}: box H {bh0:g}–{bh1:g}, alpha {ba0:g}–{ba1:g}°", fontsize=10)


save_gif(fig, update_scene, "h_alpha_scene.gif")

# 2) wetland only: mask, H-alpha of the wetland cells, Pauli RGB, and the double-bounce gain / loss
# Double-bounce part of the change, from the strongest gain (λ1) and the strongest loss (λ3) of Tc = T2 − T1:
#   ΔDB = λ1 |u1,DB|² + λ3 |u3,DB|²,   |u,DB|² = (sin α cos β)²  (double-bounce Pauli component of the eigenvector)
# > 0: double bounce increased, < 0: decreased (linear power). Cumulated over the pairs = change since the first pass.
def db_component(alpha, beta):
    return (np.sin(alpha) * np.cos(beta)) ** 2


db_delta = [np.where(wetland, p["lam1"] * db_component(p["alpha1"], p["beta1"])
                     + p["lam3"] * db_component(p["alpha3"], p["beta3"]), np.nan) for p in pairs]
db_cum = np.cumsum([np.zeros(wetland.shape)] + [np.nan_to_num(d) for d in db_delta], axis=0)
db_cum[:, ~wetland] = np.nan                                       # (passes, rows, cols), 0 on the first pass
mean_cum = [np.nanmean(c) for c in db_cum]
# map: the strong double-bounce changes of the binary map (|λ| >= db_thr), increase or decrease, in the wetland
db_changed = [np.zeros(wetland.shape, bool)] + [(g | l) & wetland for g, l in zip(db_gain, db_loss)]

vmax_wet = max(np.histogram2d(h, a, bins=BINS, range=((0, 1), (0, 90)))[0].max() for h, a in wet_pts)
fig = plt.figure(figsize=(18, 9), layout="constrained")
gs = fig.add_gridspec(2, 4, height_ratios=[3, 1], width_ratios=[1, 1.4, 1, 1])
ax_mask, ax_ha, ax_rgb, ax_db = (fig.add_subplot(gs[0, i]) for i in range(4))
ax_ts = fig.add_subplot(gs[1, :])
mask_img = np.repeat(np.nan_to_num(np.clip((span_all - g_lo) / (g_hi - g_lo), 0, 1), nan=1.0)[..., None], 3, axis=-1)
mask_grey = mask_img.copy()
mask_img[wetland] = (1.0, 0.0, 0.0)
ax_mask.imshow(mask_img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
ax_mask.set_title(f"wetland binary map (red)\nused as mask: {int(wetland.sum())} cells", fontsize=9)
ax_mask.tick_params(labelsize=6)
fig.colorbar(plt.cm.ScalarMappable(plt.Normalize(MIN_PTS, vmax_wet), "jet"), ax=ax_ha, shrink=0.7,
             label="Number of points")

# time series: general trend of the double bounce in the wetland (mean cumulative ΔDB since the first pass)
ax_ts.plot(baseline, mean_cum, "-o", color="#e0302a", lw=2, ms=5)
ax_ts.axhline(0, color="black", lw=0.5)
ax_ts.set_ylabel("mean cumulative ΔDB\n(> 0 more, < 0 less double bounce)", fontsize=8)
ax_ts.set_xticks(baseline, [f"+{b}\n{sc['date'][5:]}" for b, sc in zip(baseline, scenes)], fontsize=7)
ax_ts.set_title("double-bounce trend of the wetland since the first pass", fontsize=9)
ax_ts.grid(color="#e4e3dd", lw=0.5)
ax_ts.spines[["top", "right"]].set_visible(False)
marker = ax_ts.axvline(baseline[0], color="black", lw=1.5, ls="--")

def update_wetland(k):
    sc, b, (h, a) = scenes[k], baseline[k], wet_pts[k]
    draw_h_alpha(ax_ha, h, a, vmax_wet,
                 f"{sc['date']} (+{b} d): H-alpha of the wetland cells\nmedian H {np.median(h):.2f}, "
                 f"alpha {np.median(a):.1f}°")
    ax_rgb.clear()
    ax_rgb.imshow(pauli[k], extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
    ax_rgb.set_title(f"Pauli RGB {sc['date']}\n(R double bounce, G volume, B surface)", fontsize=9)
    ax_rgb.tick_params(labelsize=6)
    ax_db.clear()
    img = mask_grey.copy()
    img[db_changed[k]] = (1.0, 0.0, 0.0)
    ax_db.imshow(img, extent=geo["extent"], aspect=geo["px_aspect"], interpolation="nearest")
    ax_db.set_title("first pass: reference" if k == 0 else
                    f"water appearing / disappearing (red)\n{scenes[k - 1]['date'][5:]} → {sc['date'][5:]}: "
                    f"{db_changed[k].sum() * cell_km2:.0f} km²", fontsize=9)
    ax_db.tick_params(labelsize=6)
    marker.set_xdata([b, b])
    fig.suptitle(f"{roi_name} · {stack_dir.name}: wetland H-alpha and double-bounce change over time "
                 f"({wetland_rule})", fontsize=10)


save_gif(fig, update_wetland, "h_alpha_wetland.gif")